# Full Transformer Architecture: Encoder-Decoder

This notebook demonstrates the **original transformer** from "Attention Is All You Need" paper.

## Key Difference from Your Previous Notebooks:

| Architecture | Your First Notebook | Your BPE Notebook | This Notebook |
|---|---|---|---|
| **Type** | Decoder-Only | Decoder-Only | Encoder-Decoder |
| **Input** | "Patient: I" | "Patient: I" | **Two inputs:** Source + Target |
| **Attention** | Causal (← past only) | Causal (← past only) | **Encoder:** Bidirectional <br> **Decoder:** Causal <br> **Cross:** Encoder→Decoder |
| **Use Case** | Text generation | Text generation | **Translation, Summarization, Q&A** |
| **Example** | → "Patient: I have a fever" | → "Patient: I have a fever" | **Input:** "Hello world" <br> **Output:** "Hola mundo" |

## What is Encoder-Decoder?

```
SOURCE TEXT              ENCODER              DECODER            TARGET TEXT
────────────────         ───────────          ───────────         ────────────
"Hello world"  ────────► Bidirectional  ────► Causal Attn  ────► "Hola mundo"
               (tokens)   Attention           Cross-Attn           (tokens)
                          (understands)       (generates)          
```

**Encoder:** Reads and understands the entire input at once (bidirectional)
**Decoder:** Generates output one token at a time (autoregressive)
**Cross-Attention:** Decoder attends to encoder's output to see what was understood

## Setup and Imports

In [ ]:
import math
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
import json

# Reproducibility
seed = 42
torch.manual_seed(seed)

# Use GPU if available, else CPU
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

## 1. Creating a Seq2Seq Task: Text Reversal

To demonstrate the encoder-decoder transformer, we'll use a simple task: **reversing words in a sentence**.

Examples:
- Input: "patient has fever" → Output: "fever has patient"
- Input: "doctor appointment" → Output: "appointment doctor"

In [ ]:
# Create training data: word reversal task
print("Generating training data: Word Reversal")
print("="*50)

# Vocabulary
vocab = set()
training_pairs = []

# Healthcare-style vocabulary
healthcare_words = ['patient', 'doctor', 'nurse', 'hospital', 'fever', 'blood', 
             'test', 'medication', 'appointment', 'appointment', 'treatment',
             'disease', 'symptom', 'diagnosis', 'surgery', 'recovery',
             'examination', 'prescription', 'therapy', 'clinic']

# Generate pairs
import random
random.seed(42)

for _ in range(200):
    # Random sequence of 2-4 words
    num_words = random.randint(2, 4)
    words = random.choices(healthcare_words, k=num_words)
    source = ' '.join(words)
    target = ' '.join(reversed(words))
    
    training_pairs.append((source, target))
    vocab.update(words)

# Add special tokens
vocab = sorted(list(vocab))
special_tokens = ['<pad>', '<sos>', '<eos>', '<unk>']
vocab = special_tokens + vocab
vocab_size = len(vocab)

print(f"✓ Generated {len(training_pairs)} training pairs")
print(f"✓ Vocabulary size: {vocab_size}")
print(f"\nExample pairs:")
for i in range(3):
    print(f"  Input:  {training_pairs[i][0]}")
    print(f"  Output: {training_pairs[i][1]}")
    print()

# Create mappings
stoi = {ch: i for i, ch in enumerate(vocab)}
itos = {i: ch for i, ch in enumerate(vocab)}

print(f"\nVocabulary: {vocab[:10]}...") # Show first 10

## 2. Tokenization and Data Processing

In [ ]:
def encode(text):
    """Convert text to token IDs (word-level tokenization)"""
    words = text.split()
    return [stoi.get(word, stoi['<unk>']) for word in words]

def decode(indices):
    """Convert token IDs back to text"""
    return ' '.join([itos[i] for i in indices if i not in [stoi['<pad>'], stoi['<sos>'], stoi['<eos>']]])

# Test encoding/decoding
test_input = "patient has fever"
encoded = encode(test_input)
decoded = decode(encoded)

print(f"Original: {test_input}")
print(f"Encoded: {encoded}")
print(f"Decoded: {decoded}")

In [ ]:
# Hyperparameters
batch_size = 16
max_seq_len = 10  # Max words in a sequence
max_iters = 500
learning_rate = 1e-3
eval_interval = 50
eval_iters = 20
n_embd = 64
n_head = 4
n_layer = 2
dropout = 0.1

# Prepare data
encoded_pairs = [(encode(src), encode(tgt)) for src, tgt in training_pairs]

# Train/val split
n_train = int(0.8 * len(encoded_pairs))
train_pairs = encoded_pairs[:n_train]
val_pairs = encoded_pairs[n_train:]

print(f"Train pairs: {len(train_pairs)}")
print(f"Val pairs: {len(val_pairs)}")

def pad_sequence(seq, max_len, pad_idx=0):
    """Pad or truncate sequence to max_len"""
    if len(seq) >= max_len:
        return seq[:max_len]
    return seq + [pad_idx] * (max_len - len(seq))

def get_batch(split='train'):
    """Get a batch of data"""
    pairs = train_pairs if split == 'train' else val_pairs
    indices = torch.randint(len(pairs), (batch_size,))
    
    src_batch = []
    tgt_batch = []
    
    for idx in indices:
        src, tgt = pairs[idx]
        # Add SOS/EOS tokens
        src = [stoi['<sos>']] + src + [stoi['<eos>']]
        tgt = [stoi['<sos>']] + tgt + [stoi['<eos>']]
        
        src_batch.append(pad_sequence(src, max_seq_len))
        tgt_batch.append(pad_sequence(tgt, max_seq_len))
    
    src_batch = torch.tensor(src_batch, dtype=torch.long, device=DEVICE)
    tgt_batch = torch.tensor(tgt_batch, dtype=torch.long, device=DEVICE)
    
    return src_batch, tgt_batch

# Test batch
src, tgt = get_batch('train')
print(f"\nBatch shapes: src={src.shape}, tgt={tgt.shape}")
print(f"Example src: {src[0].tolist()}")
print(f"Example tgt: {tgt[0].tolist()}")

## 3. Transformer Architecture: ENCODER

**Encoder Key Features:**
- ✅ **Bidirectional attention:** Each position can attend to ALL previous AND future positions
- ✅ **No causal masking:** The model sees the entire input sequence at once
- ✅ **Produces context vectors:** Rich representations that capture the full input meaning

The encoder's job is: **"Understand and represent the input"**

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multiple attention heads"""
    def __init__(self, num_heads, head_size, is_causal=False):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, is_causal=is_causal) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        out = torch.cat([h(x, mask=mask) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

## 4. Transformer Architecture: DECODER

**Decoder Key Features:**
- ✅ **Causal (self-)attention:** Can only attend to current and PAST tokens (autoregressive)
- ✅ **Cross-attention:** Attends to encoder output to see what was understood
- ✅ **Generates one token at a time:** Uses previously generated tokens as input

The decoder's job is: **"Generate output while looking at the encoder's understanding"**

In [ ]:
class CrossAttentionHead(nn.Module):
    """Cross-attention head: Decoder attends to encoder"""
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q_x, kv_x, mask=None):
        """q_x: decoder state, kv_x: encoder output"""
        k = self.key(kv_x)
        q = self.query(q_x)
        v = self.value(kv_x)
        
        wei = q @ k.transpose(-2, -1) * (q.shape[-1]**-0.5)
        
        if mask is not None:
            wei = wei.masked_fill(mask == 0, float('-inf'))
        
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v
        return out

class MultiHeadCrossAttention(nn.Module):
    """Multiple cross-attention heads"""
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([CrossAttentionHead(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q_x, kv_x, mask=None):
        out = torch.cat([h(q_x, kv_x, mask=mask) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

class DecoderBlock(nn.Module):
    """Decoder block: Self-attention + Cross-attention + Feed-forward"""
    def __init__(self):
        super().__init__()
        head_size = n_embd // n_head
        
        # Self-attention (causal)
        self.self_attn = MultiHeadAttention(n_head, head_size, is_causal=True)  # ← CAUSAL
        
        # Cross-attention to encoder
        self.cross_attn = MultiHeadCrossAttention(n_head, head_size)  # ← CROSS-ATTENTION
        
        self.ffwd = FeedForward()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ln3 = nn.LayerNorm(n_embd)

    def forward(self, x, encoder_out, self_mask=None, cross_mask=None):
        # Self-attention (look at own previous tokens)
        x = x + self.self_attn(self.ln1(x), mask=self_mask)
        
        # Cross-attention (look at encoder output)
        x = x + self.cross_attn(self.ln2(x), encoder_out, mask=cross_mask)
        
        # Feed-forward
        x = x + self.ffwd(self.ln3(x))
        return x

class Decoder(nn.Module):
    """Stack of decoder blocks with cross-attention"""
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, n_embd)
        self.pos_embedding = nn.Embedding(max_seq_len, n_embd)
        self.blocks = nn.Sequential(*[DecoderBlock() for _ in range(n_layer)])
        self.ln = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, x, encoder_out, self_mask=None, cross_mask=None):
        B, T = x.shape
        
        # Embeddings
        tok_emb = self.embedding(x)
        pos_emb = self.pos_embedding(torch.arange(T, device=DEVICE))
        x = tok_emb + pos_emb
        
        # Pass through all decoder blocks
        for block in self.blocks:
            x = block(x, encoder_out, self_mask=self_mask, cross_mask=cross_mask)
        
        x = self.ln(x)
        logits = self.lm_head(x)
        return logits

print("✓ Decoder architecture defined")

## 5. Complete Transformer Model

Combines encoder and decoder with the full training loop.

In [ ]:
class Transformer(nn.Module):
    """Full Encoder-Decoder Transformer"""
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        # Encode source
        encoder_out = self.encoder(src, mask=src_mask)
        
        # Decode target (teacher forcing: use ground truth)
        logits = self.decoder(tgt, encoder_out, self_mask=tgt_mask, cross_mask=src_mask)
        
        return logits
    
    def generate(self, src, max_len=10, src_mask=None):
        """Generate output sequence given input sequence"""
        with torch.no_grad():
            # Encode source once
            encoder_out = self.encoder(src, mask=src_mask)
            
            # Start with SOS token
            B = src.shape[0]
            generated = torch.full((B, 1), stoi['<sos>'], dtype=torch.long, device=DEVICE)
            
            # Generate one token at a time
            for _ in range(max_len):
                # Decode next token
                logits = self.decoder(generated, encoder_out, self_mask=None, cross_mask=src_mask)
                
                # Get last token prediction
                next_logits = logits[:, -1, :]
                next_probs = F.softmax(next_logits, dim=-1)
                
                # Sample or greedy
                next_token = torch.argmax(next_probs, dim=-1, keepdim=True)
                
                # Append to sequence
                generated = torch.cat([generated, next_token], dim=1)
                
                # Stop if EOS token
                if (next_token == stoi['<eos>']).all():
                    break
            
            return generated

# Create model
model = Transformer().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Transformer model created")
print(f"  Encoder parameters: {sum(p.numel() for p in model.encoder.parameters()):,}")
print(f"  Decoder parameters: {sum(p.numel() for p in model.decoder.parameters()):,}")
print(f"  Total parameters: {total_params:,}")

## 6. Training Loop

In [ ]:
def create_mask(seq, pad_idx):
    """Create attention mask for padding positions
    Returns shape (B, 1, T) for broadcasting with attention scores (B, T, T)
    """
    # seq shape: (B, T)
    mask = (seq != pad_idx).unsqueeze(1)  # (B, 1, T) - broadcasts to (B, T, T)
    return mask

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            src, tgt = get_batch(split)
            
            # Create masks
            src_mask = create_mask(src, stoi['<pad>'])
            tgt_input = tgt[:, :-1]
            tgt_target = tgt[:, 1:]
            tgt_mask = create_mask(tgt_input, stoi['<pad>'])
            
            # Forward pass
            logits = model(src, tgt_input, src_mask=src_mask, tgt_mask=tgt_mask)
            
            # Loss (use reshape instead of view for compatibility)
            loss = F.cross_entropy(logits.reshape(-1, vocab_size), tgt_target.reshape(-1))
            losses[k] = loss.item()
        
        out[split] = losses.mean()
    
    model.train()
    return out

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Training
train_losses = []
val_losses = []

print("Starting training...")
model.train()

for iter in range(max_iters):
    # Evaluate
    if iter % eval_interval == 0:
        losses = estimate_loss()
        train_losses.append(losses['train'].item())
        val_losses.append(losses['val'].item())
        print(f"Step {iter:3d}: train loss = {losses['train']:.4f}, val loss = {losses['val']:.4f}")
    
    # Training step
    src, tgt = get_batch('train')
    
    # Masks
    src_mask = create_mask(src, stoi['<pad>'])
    tgt_input = tgt[:, :-1]
    tgt_target = tgt[:, 1:]
    tgt_mask = create_mask(tgt_input, stoi['<pad>'])
    
    # Forward
    logits = model(src, tgt_input, src_mask=src_mask, tgt_mask=tgt_mask)
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), tgt_target.reshape(-1))
    
    # Backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(f"\n✓ Training complete!")
print(f"  Final train loss: {train_losses[-1]:.4f}")
print(f"  Final val loss: {val_losses[-1]:.4f}")

In [ ]:
# COMPLETE TRANSFORMER WITH ALL FIXED COMPONENTS

# Helper class
class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

# Single attention head
class HeadFixed(nn.Module):
    """Single attention head - FIXED VERSION"""
    def __init__(self, head_size, is_causal=False):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.is_causal = is_causal
        
        if is_causal:
            self.register_buffer('tril', torch.tril(torch.ones(max_seq_len, max_seq_len)))
        
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        B, T, C = x.shape
        head_size = self.query(x).shape[-1]
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        
        wei = q @ k.transpose(-2, -1) * (head_size**-0.5)
        
        if self.is_causal:
            wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        
        if mask is not None:
            wei = wei.masked_fill(mask == 0, float('-inf'))
        
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v
        return out

# Multi-head attention
class MultiHeadAttentionFixed(nn.Module):
    """Multiple attention heads - FIXED VERSION"""
    def __init__(self, num_heads, head_size, is_causal=False):
        super().__init__()
        self.heads = nn.ModuleList([HeadFixed(head_size, is_causal=is_causal) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        out = torch.cat([h(x, mask=mask) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

# Cross-attention components
class CrossAttentionHead(nn.Module):
    """Cross-attention head: Decoder attends to encoder"""
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q_x, kv_x, mask=None):
        """q_x: decoder state, kv_x: encoder output"""
        k = self.key(kv_x)
        q = self.query(q_x)
        v = self.value(kv_x)
        
        wei = q @ k.transpose(-2, -1) * (q.shape[-1]**-0.5)
        
        if mask is not None:
            wei = wei.masked_fill(mask == 0, float('-inf'))
        
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v
        return out

class MultiHeadCrossAttention(nn.Module):
    """Multiple cross-attention heads"""
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([CrossAttentionHead(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q_x, kv_x, mask=None):
        out = torch.cat([h(q_x, kv_x, mask=mask) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

# Encoder components
class EncoderBlockFixed(nn.Module):
    """Encoder block - FIXED VERSION"""
    def __init__(self):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttentionFixed(n_head, head_size, is_causal=False)
        self.ffwd = FeedForward()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x, mask=None):
        x = x + self.sa(self.ln1(x), mask=mask)
        x = x + self.ffwd(self.ln2(x))
        return x

class EncoderFixed(nn.Module):
    """Stack of encoder blocks - FIXED VERSION"""
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, n_embd)
        self.pos_embedding = nn.Embedding(max_seq_len, n_embd)
        self.blocks = nn.Sequential(*[EncoderBlockFixed() for _ in range(n_layer)])
        self.ln = nn.LayerNorm(n_embd)

    def forward(self, x, mask=None):
        B, T = x.shape
        tok_emb = self.embedding(x)
        pos_emb = self.pos_embedding(torch.arange(T, device=DEVICE))
        x = tok_emb + pos_emb
        
        for block in self.blocks:
            x = block(x, mask=mask)
        
        x = self.ln(x)
        return x

# Decoder components
class DecoderBlockFixed(nn.Module):
    """Decoder block - FIXED VERSION"""
    def __init__(self):
        super().__init__()
        head_size = n_embd // n_head
        
        self.self_attn = MultiHeadAttentionFixed(n_head, head_size, is_causal=True)
        self.cross_attn = MultiHeadCrossAttention(n_head, head_size)
        
        self.ffwd = FeedForward()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ln3 = nn.LayerNorm(n_embd)

    def forward(self, x, encoder_out, self_mask=None, cross_mask=None):
        x = x + self.self_attn(self.ln1(x), mask=self_mask)
        x = x + self.cross_attn(self.ln2(x), encoder_out, mask=cross_mask)
        x = x + self.ffwd(self.ln3(x))
        return x

class DecoderFixed(nn.Module):
    """Stack of decoder blocks - FIXED VERSION"""
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, n_embd)
        self.pos_embedding = nn.Embedding(max_seq_len, n_embd)
        self.blocks = nn.Sequential(*[DecoderBlockFixed() for _ in range(n_layer)])
        self.ln = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size)

    def forward(self, x, encoder_out, self_mask=None, cross_mask=None):
        B, T = x.shape
        
        tok_emb = self.embedding(x)
        pos_emb = self.pos_embedding(torch.arange(T, device=DEVICE))
        x = tok_emb + pos_emb
        
        for block in self.blocks:
            x = block(x, encoder_out, self_mask=self_mask, cross_mask=cross_mask)
        
        x = self.ln(x)
        logits = self.head(x)
        return logits

# Complete Transformer with generate method
class TransformerFixed(nn.Module):
    """Encoder-Decoder Transformer - FIXED VERSION"""
    def __init__(self):
        super().__init__()
        self.encoder = EncoderFixed()
        self.decoder = DecoderFixed()

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        encoder_out = self.encoder(src, mask=src_mask)
        logits = self.decoder(tgt, encoder_out, self_mask=tgt_mask, cross_mask=src_mask)
        return logits
    
    def generate(self, src, max_len=10, src_mask=None):
        """Generate output sequence given input sequence"""
        with torch.no_grad():
            # Encode source once
            encoder_out = self.encoder(src, mask=src_mask)
            
            # Start with SOS token
            B = src.shape[0]
            generated = torch.full((B, 1), stoi['<sos>'], dtype=torch.long, device=DEVICE)
            
            # Generate one token at a time
            for _ in range(max_len):
                # Decode next token
                logits = self.decoder(generated, encoder_out, self_mask=None, cross_mask=src_mask)
                
                # Get last token prediction
                next_logits = logits[:, -1, :]
                next_probs = F.softmax(next_logits, dim=-1)
                
                # Sample or greedy
                next_token = torch.argmax(next_probs, dim=-1, keepdim=True)
                
                # Append to sequence
                generated = torch.cat([generated, next_token], dim=1)
                
                # Stop if EOS token
                if (next_token == stoi['<eos>']).all():
                    break
            
            return generated

# Create fresh model with fixed classes
model = TransformerFixed().to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Fixed Transformer model created with {total_params:,} parameters")
print(f"  Model has generate() method: {hasattr(model, 'generate')}")

## 7. Loss Visualisation

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(len(train_losses)), train_losses, label='Train Loss', marker='o', markersize=5)
plt.plot(range(len(val_losses)), val_losses, label='Validation Loss', marker='s', markersize=5)
plt.xlabel('Eval Step')
plt.ylabel('Loss')
plt.title('Encoder-Decoder Transformer: Training vs Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Testing the Model

Let's test if the model learned to reverse words!

In [ ]:
print("TESTING ENCODER-DECODER TRANSFORMER")
print("="*60)
print("Task: Reverse the order of words in a sentence")
print()

# Test examples
test_inputs = [
    "patient has fever",
    "doctor appointment test",
    "hospital blood treatment",
]

model.eval()

for test_input in test_inputs:
    # Encode input
    src_tokens = encode(test_input)
    src_padded = pad_sequence(src_tokens, max_seq_len, stoi['<pad>'])
    src_tensor = torch.tensor([src_padded], dtype=torch.long, device=DEVICE)
    
    # Create mask
    src_mask = create_mask(src_tensor, stoi['<pad>'])
    
    # Generate output
    output_tokens = model.generate(src_tensor, max_len=max_seq_len, src_mask=src_mask)
    output_text = decode(output_tokens[0].cpu().tolist())
    
    # Expected output
    expected = ' '.join(reversed(test_input.split()))
    
    # Check if correct
    is_correct = '✓' if output_text == expected else '✗'
    
    print(f"{is_correct} Input:    {test_input}")
    print(f"  Output:   {output_text}")
    print(f"  Expected: {expected}")
    print()

## 9. Understanding Encoder vs Decoder Attention

Let's visualize the three types of attention in action:

In [ ]:
print("\n" + "="*70)
print("UNDERSTANDING ENCODER-DECODER ATTENTION")
print("="*70)

attention_types = {
    'Encoder Self-Attention': {
        'Query': 'Decoder position',
        'Key/Value': 'Encoder positions (ALL)',
        'Masking': 'Padding positions only',
        'Can see': '✓ Past, ✓ Current, ✓ Future',
        'Purpose': 'Understand full input',
        'Example': 'When encoding "fever has patient", each word can see all other words'
    },
    'Decoder Self-Attention': {
        'Query': 'Decoder position',
        'Key/Value': 'Decoder positions (ONLY PAST)',
        'Masking': 'Padding + Causal (future)',
        'Can see': '✓ Past, ✓ Current, ✗ Future',
        'Purpose': 'Remember what was generated',
        'Example': 'When generating token 3, only see tokens 1,2,3 (not future)'
    },
    'Cross-Attention': {
        'Query': 'Decoder position',
        'Key/Value': 'Encoder output (ALL)',
        'Masking': 'Padding only',
        'Can see': '✓ All encoder positions',
        'Purpose': 'Look at encoder understanding',
        'Example': 'Decoder looks at all encoded words to decide what to generate'
    }
}

for attn_type, details in attention_types.items():
    print(f"\n🔍 {attn_type}")
    for key, value in details.items():
        print(f"   {key}: {value}")

## 10. Comparing with Previous Architectures

In [ ]:
print("\n" + "="*70)
print("ARCHITECTURE COMPARISON: What You've Learned")
print("="*70)

comparison = """
┌─────────────────────────────────────────────────────────────────────────┐
│  NOTEBOOK 1 & 2: DECODER-ONLY (GPT-Style)                              │
├─────────────────────────────────────────────────────────────────────────┤
│  Architecture:  [Embeddings] → [Causal Blocks] → [Output Layer]        │
│  Attention:     Only PAST tokens (causal masking)                       │
│  Task:          Text Generation ("Patient: I" → "...have a fever")     │
│  Use Cases:     Language models, chatbots, creative writing             │
│  Training:      Autoregressive (predict next token)                     │
│  Inference:     Generate one token at a time                            │
│  Examples:      GPT-2, GPT-3, GPT-4, LLaMA                             │
│                                                                          │
│  ✓ Simple and efficient                                                  │
│  ✓ Good for open-ended generation                                        │
│  ✗ Can't "look ahead" (sees only past)                                  │
│  ✗ Not good for translation/summarization (needs full input)            │
└─────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────┐
│  NOTEBOOK 3 (THIS ONE): ENCODER-DECODER (Transformer-Style)            │
├─────────────────────────────────────────────────────────────────────────┤
│  Architecture:  [Encoder Blocks] → [Decoder Blocks] → [Output Layer]   │
│  Attention:     Encoder: sees ALL positions (bidirectional)             │
│                 Decoder: sees only PAST (causal)                        │
│                 Cross: Decoder looks at encoder                         │
│  Task:          Seq2Seq (Word Reversal, Translation, Summarization)    │
│  Use Cases:     Translation, summarization, question-answering          │
│  Training:      Teacher forcing (feed true targets)                     │
│  Inference:     Generate one token, use encoder context                │
│  Examples:      T5, BART, mBART, original Transformer                  │
│                                                                          │
│  ✓ Can "look ahead" (encoder sees full input)                           │
│  ✓ Excellent for translation/summarization                              │
│  ✓ Clean separation: understand (encoder) vs generate (decoder)         │
│  ✗ Slightly more complex                                                │
│  ✗ Requires source AND target during training                           │
└─────────────────────────────────────────────────────────────────────────┘

VISUAL COMPARISON:

DECODER-ONLY (GPT):              ENCODER-DECODER (This):
─────────────────────────────────────────────────────────

Input: "Patient: I"     Input: "patient fever has"
  ↓                         ↓
[Causal Blocks]         [Encoder Blocks] ← Sees FULL input
  ↓                         ↓
Attend to: Past         [Decoder Blocks] ← Generates output
  ↓                         ↓
Generate: "have"        Attend to: Encoder + Past
  ↓                         ↓
"have a fever"          Output: "has fever patient"

"""

print(comparison)

## 11. Summary

You now understand all three transformer variants:

1. **Decoder-Only (Notebooks 1 & 2)**: Used by GPT models
   - Good for open-ended text generation
   - Generates text one token at a time
   - Sees only past context

2. **Encoder-Decoder (This Notebook)**: Used by T5, BART, original Transformer
   - Good for understanding input + generating output
   - Encoder processes full input (bidirectional)
   - Decoder generates while reading encoder's understanding
   - Perfect for translation, summarization, Q&A

3. **Key Differences**:
   - Encoder can see the entire input at once
   - Decoder generates autoregressively (one token at a time)
   - Cross-attention bridges encoder and decoder
   - Different masking strategies (causal vs bidirectional)

**Next steps:**
- Try increasing `vocab_size` for more words
- Change the task (e.g., paraphrasing instead of reversal)
- Train longer and measure accuracy
- Compare generation quality between encoder-decoder vs decoder-only